# Day 03 — Data Types and Data Quality

**Objective:** Investigate missing values, duplicate records, data types, unusual values and potential outliers in the three datasets.

Update the CSV paths below. The Student Performance file is semicolon-separated; the other two are comma-separated.

This notebook flags records for investigation. It does not automatically delete or alter data.

## Section 1 — Load the datasets

In [1]:
import pandas as pd
import numpy as np

student_path = "student-por.csv"
houses_path = "nigeria_houses_data.csv"
stroke_path = "StrokeDisease.csv"

student_df = pd.read_csv(student_path, sep=";")
houses_df = pd.read_csv(houses_path)
stroke_df = pd.read_csv(stroke_path)

datasets = {
    "Student Performance": student_df,
    "Nigeria House Prices": houses_df,
    "Stroke Disease": stroke_df
}

print("All three datasets loaded successfully.")

All three datasets loaded successfully.


## Section 2 — Missing values

In [2]:
missing_rows = []

for name, df in datasets.items():
    for col in df.columns:
        count = int(df[col].isna().sum())
        missing_rows.append({
            "Dataset": name,
            "Column": col,
            "Missing Values": count,
            "Missing (%)": round(count / len(df) * 100, 2)
        })

missing_detail = pd.DataFrame(missing_rows)
display(missing_detail)

missing_summary = pd.DataFrame([
    {
        "Dataset": name,
        "Total Missing Values": int(df.isna().sum().sum()),
        "Columns With Missing Values": int(df.isna().any().sum())
    }
    for name, df in datasets.items()
])
display(missing_summary)

,Dataset,Column,Missing Values,Missing (%)
0,Student Performance,school,0,0.00
1,Student Performance,sex,0,0.00
2,Student Performance,age,0,0.00
3,Student Performance,address,0,0.00
4,Student Performance,famsize,0,0.00
5,Student Performance,Pstatus,0,0.00
6,Student Performance,Medu,0,0.00
7,Student Performance,Fedu,0,0.00
8,Student Performance,Mjob,0,0.00
9,Student Performance,Fjob,0,0.00


,Dataset,Total Missing Values,Columns With Missing Values
0,Student Performance,0,0
1,Nigeria House Prices,0,0
2,Stroke Disease,201,1


## Section 3 — Duplicate records

In [3]:
duplicate_report = pd.DataFrame([
    {
        "Dataset": name,
        "Duplicate Records": int(df.duplicated().sum()),
        "Duplicate (%)": round(df.duplicated().mean() * 100, 2)
    }
    for name, df in datasets.items()
])
display(duplicate_report)

,Dataset,Duplicate Records,Duplicate (%)
0,Student Performance,0,0.00
1,Nigeria House Prices,10438,42.91
2,Stroke Disease,0,0.00


In [4]:
# Display duplicate rows for inspection (if any)
for name, df in datasets.items():
    print("\\n", name, "— duplicate rows")
    display(df[df.duplicated(keep=False)].head(20))

\n Student Performance — duplicate rows


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3


\n Nigeria House Prices — duplicate rows


,bedrooms,bathrooms,toilets,parking_space,title,town,state,price
2,4.0,5.0,5.0,4.0,Detached Duplex,Lekki,Lagos,120000000.0
4,4.0,4.0,5.0,2.0,Semi Detached Duplex,Lekki,Lagos,75000000.0
5,5.0,5.0,6.0,1.0,Detached Duplex,Lekki,Lagos,450000000.0
6,4.0,5.0,5.0,4.0,Detached Duplex,Lekki,Lagos,65000000.0
9,4.0,4.0,5.0,5.0,Detached Duplex,Ajah,Lagos,60000000.0
12,4.0,5.0,6.0,3.0,Semi Detached Duplex,Lekki,Lagos,65000000.0
13,4.0,4.0,5.0,4.0,Semi Detached Duplex,Lekki,Lagos,70000000.0
14,6.0,6.0,7.0,4.0,Detached Duplex,Victoria Island (VI),Lagos,450000000.0
16,4.0,4.0,5.0,4.0,Terraced Duplexes,Lekki,Lagos,48000000.0
18,4.0,5.0,5.0,4.0,Detached Duplex,Victoria Island (VI),Lagos,170000000.0


\n Stroke Disease — duplicate rows


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke


## Section 4 — Data types and feature categories

In [5]:
dtype_rows = []

for name, df in datasets.items():
    for col in df.columns:
        dtype_rows.append({
            "Dataset": name,
            "Column": col,
            "Stored Data Type": str(df[col].dtype),
            "Unique Values": int(df[col].nunique(dropna=True)),
            "Missing Values": int(df[col].isna().sum())
        })

dtype_report = pd.DataFrame(dtype_rows)
display(dtype_report)

,Dataset,Column,Stored Data Type,Unique Values,Missing Values
0,Student Performance,school,object,2,0
1,Student Performance,sex,object,2,0
2,Student Performance,age,int64,8,0
3,Student Performance,address,object,2,0
4,Student Performance,famsize,object,2,0
5,Student Performance,Pstatus,object,2,0
6,Student Performance,Medu,int64,5,0
7,Student Performance,Fedu,int64,5,0
8,Student Performance,Mjob,object,5,0
9,Student Performance,Fjob,object,5,0


In [6]:
feature_rows = []

for name, df in datasets.items():
    numeric = df.select_dtypes(include=["number"]).columns.tolist()
    categorical = df.select_dtypes(include=["object", "string", "category"]).columns.tolist()
    boolean = df.select_dtypes(include=["bool"]).columns.tolist()

    feature_rows.append({
        "Dataset": name,
        "Numerical Features": len(numeric),
        "Categorical Features": len(categorical),
        "Boolean Features": len(boolean),
        "Numerical Column Names": ", ".join(numeric),
        "Categorical Column Names": ", ".join(categorical)
    })

feature_summary = pd.DataFrame(feature_rows)
display(feature_summary)

,Dataset,Numerical Features,Categorical Features,Boolean Features,Numerical Column Names,Categorical Column Names
0,Student Performance,16,17,0,"age, Medu, Fedu, traveltime, studytime, failur...","school, sex, address, famsize, Pstatus, Mjob, ..."
1,Nigeria House Prices,5,3,0,"bedrooms, bathrooms, toilets, parking_space, p...","title, town, state"
2,Stroke Disease,7,5,0,"id, age, hypertension, heart_disease, avg_gluc...","gender, ever_married, work_type, Residence_typ..."


**Interpretation note:** These counts are based on Pandas storage types. Integer-coded variables may represent categories or ordinal ratings even when stored as numbers. Check each column's meaning before modelling.

## Section 5 — Unusual or potentially invalid values

In [7]:
# Define simple plausibility checks for expected columns.
# These are investigation rules, not universal truths.

range_rules = {
    "Student Performance": {
        "age": (15, 22),
        "G1": (0, 20), "G2": (0, 20), "G3": (0, 20),
        "absences": (0, None)
    },
    "Nigeria House Prices": {
        "bedrooms": (0, None), "bathrooms": (0, None),
        "toilets": (0, None), "parking_space": (0, None),
        "price": (0, None)
    },
    "Stroke Disease": {
        "age": (0, 120), "bmi": (0, None),
        "avg_glucose_level": (0, None),
        "hypertension": (0, 1), "heart_disease": (0, 1),
        "stroke": (0, 1)
    }
}

unusual_rows = []

for name, df in datasets.items():
    for col, (minimum, maximum) in range_rules[name].items():
        if col not in df.columns:
            continue

        values = pd.to_numeric(df[col], errors="coerce")
        mask = pd.Series(False, index=df.index)

        if minimum is not None:
            mask |= values.notna() & (values < minimum)
        if maximum is not None:
            mask |= values.notna() & (values > maximum)

        unusual_rows.append({
            "Dataset": name,
            "Column": col,
            "Expected Minimum": minimum,
            "Expected Maximum": maximum,
            "Potentially Unusual Values": int(mask.sum())
        })

unusual_report = pd.DataFrame(unusual_rows)
display(unusual_report)

,Dataset,Column,Expected Minimum,Expected Maximum,Potentially Unusual Values
0,Student Performance,age,15,22.0,0
1,Student Performance,G1,0,20.0,0
2,Student Performance,G2,0,20.0,0
3,Student Performance,G3,0,20.0,0
4,Student Performance,absences,0,NaN,0
5,Nigeria House Prices,bedrooms,0,NaN,0
6,Nigeria House Prices,bathrooms,0,NaN,0
7,Nigeria House Prices,toilets,0,NaN,0
8,Nigeria House Prices,parking_space,0,NaN,0
9,Nigeria House Prices,price,0,NaN,0


## Section 6 — Potential numerical outliers (IQR method)

In [8]:
outlier_rows = []

for name, df in datasets.items():
    numeric_df = df.select_dtypes(include=["number"])

    for col in numeric_df.columns:
        values = numeric_df[col].dropna()
        if len(values) < 4:
            continue

        q1 = values.quantile(0.25)
        q3 = values.quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        count = int(((values < lower) | (values > upper)).sum())

        outlier_rows.append({
            "Dataset": name,
            "Numerical Feature": col,
            "Q1": round(q1, 3),
            "Q3": round(q3, 3),
            "IQR": round(iqr, 3),
            "Lower Bound": round(lower, 3),
            "Upper Bound": round(upper, 3),
            "Potential Outliers": count,
            "Outliers (%)": round(count / len(values) * 100, 2)
        })

outlier_report = pd.DataFrame(outlier_rows)
display(outlier_report)

,Dataset,Numerical Feature,Q1,Q3,IQR,Lower Bound,Upper Bound,Potential Outliers,Outliers (%)
0,Student Performance,age,1.600000e+01,1.800000e+01,2.000000e+00,1.300000e+01,2.100000e+01,1,0.15
1,Student Performance,Medu,2.000000e+00,4.000000e+00,2.000000e+00,-1.000000e+00,7.000000e+00,0,0.00
2,Student Performance,Fedu,1.000000e+00,3.000000e+00,2.000000e+00,-2.000000e+00,6.000000e+00,0,0.00
3,Student Performance,traveltime,1.000000e+00,2.000000e+00,1.000000e+00,-5.000000e-01,3.500000e+00,16,2.47
4,Student Performance,studytime,1.000000e+00,2.000000e+00,1.000000e+00,-5.000000e-01,3.500000e+00,35,5.39
5,Student Performance,failures,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,100,15.41
6,Student Performance,famrel,4.000000e+00,5.000000e+00,1.000000e+00,2.500000e+00,6.500000e+00,51,7.86
7,Student Performance,freetime,3.000000e+00,4.000000e+00,1.000000e+00,1.500000e+00,5.500000e+00,45,6.93
8,Student Performance,goout,2.000000e+00,4.000000e+00,2.000000e+00,-1.000000e+00,7.000000e+00,0,0.00
9,Student Performance,Dalc,1.000000e+00,2.000000e+00,1.000000e+00,-5.000000e-01,3.500000e+00,34,5.24


## Section 7 — Main Data Quality Report

In [9]:
# Summary table in the requested format
quality_report = pd.DataFrame([
    {
        "Dataset": name,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Missing Values": int(df.isna().sum().sum()),
        "Duplicates": int(df.duplicated().sum()),
        "Numerical Features": len(df.select_dtypes(include=["number"]).columns),
        "Categorical Features": len(df.select_dtypes(include=["object", "string", "category"]).columns)
    }
    for name, df in datasets.items()
])

display(quality_report)

,Dataset,Rows,Columns,Missing Values,Duplicates,Numerical Features,Categorical Features
0,Student Performance,649,33,0,0,16,17
1,Nigeria House Prices,24326,8,0,10438,5,3
2,Stroke Disease,5110,12,201,0,7,5


## Section 8 — Export the report tables

In [10]:
quality_report.to_csv("Day03_Data_Quality_Summary.csv", index=False)
missing_detail.to_csv("Day03_Missing_Values_Detail.csv", index=False)
duplicate_report.to_csv("Day03_Duplicates.csv", index=False)
dtype_report.to_csv("Day03_Data_Types.csv", index=False)
unusual_report.to_csv("Day03_Unusual_Values.csv", index=False)
outlier_report.to_csv("Day03_Potential_Outliers.csv", index=False)

print("All Day 3 report tables have been saved as CSV files.")

All Day 3 report tables have been saved as CSV files.


## Section 9 — Your written findings

After running the notebook, answer these questions using your actual results:

1. Which dataset has the most missing values?
2. Which dataset has the most duplicate records?
3. Which columns have data types that may need review?
4. Which values were flagged as potentially unusual?
5. Which numerical features have the most potential outliers?
6. Are the flagged outliers errors, or could they be valid observations?
7. What cleaning actions would you recommend, and why?

Do not automatically delete duplicates or outliers. Investigate them first.

## Day 3 completion checklist

- [ ] Checked missing values by dataset and column.
- [ ] Checked duplicate records.
- [ ] Inspected stored data types.
- [ ] Counted numerical and categorical features.
- [ ] Checked potentially unusual values.
- [ ] Detected potential outliers using IQR.
- [ ] Created the main Data Quality Report table.
- [ ] Exported the report tables.
- [ ] Wrote findings based on the actual outputs.
- [ ] Saved this notebook as **Day03_Data_Quality.ipynb**.

**Suggested GitHub folder:** `Day03_Data_Quality/`

**Suggested commit message:** `Complete Day 3 data quality assessment for all three datasets`

**Reminder:** The IQR method flags statistical extremes, not necessarily mistakes. Integer-coded categories and ordinal variables may also need manual review.